In [0]:
dbutils.widgets.text("catalog", "claims")   # target catalog (bundle job passes dev/prod catalog)
catalog = dbutils.widgets.get("catalog")


# publish_analytics — R1 star schema, built OUTSIDE the SDP pipeline

**Why outside the pipeline (same pattern as `publish_gold`):** the fact's main source,
`claims.gold.claims_enriched`, is a batch table built by the `publish_gold` notebook
(`claims_daily` job) — it is NOT a pipeline dataset, so the analytics layer cannot depend
on pipeline-managed reads of it.

**What it builds** (batch tables in `claims.analytics`, one row per contract):
- `claims.analytics.dim_date` — calendar 2025-07-01..2026-06-30 (365 rows)
- `claims.analytics.dim_member` — member dim, NO PII columns
- `claims.analytics.dim_hospital` — hospital dim
- `claims.analytics.dim_plan` — plan attributes per POLICY VERSION (SCD2 windows; valid_to empty = open)
- `claims.analytics.fact_claims` — one row per claim, CLUSTER BY (admission_date_key);
  plan_sk = plan AT ADMISSION (never the current plan); settled_date_key empty until settled

**Sources:** `claims.silver.members`, `claims.silver.hospitals`, `claims.silver.plans`,
`claims.silver.dim_policy_scd2` (pipeline-published, read with qualified names) and
`claims.gold.claims_enriched` (batch-built by publish_gold).

**Keys:** surrogate keys are BIGINT, unique, never empty. Primary keys and foreign keys are
added AFTER the CTAS builds (informational constraints; CTAS cannot declare them inline).
Run this notebook only AFTER the pipeline (bronze+silver) and `publish_gold` have succeeded.


In [0]:
%sql
-- R1: claims.analytics.dim_date — calendar dimension, one row per calendar day
-- from 2025-07-01 to 2026-06-30 (the FY window). week_start = Monday of that week.
-- Batch CTAS; constraints added in the 'Add named PK/FK constraints' cell.
CREATE OR REPLACE TABLE IDENTIFIER(:catalog || '.analytics.dim_date')
COMMENT 'R1: calendar dimension, one row per day 2025-07-01..2026-06-30; date_key = yyyymmdd'
AS
SELECT
  CAST(date_format(d, 'yyyyMMdd') AS INT)           AS date_key,
  d                                                 AS full_date,
  year(d)                                           AS year,
  quarter(d)                                        AS quarter,
  month(d)                                          AS month,
  CAST(date_trunc('month', d) AS DATE)              AS month_start,
  CAST(date_sub(d, (dayofweek(d) + 5) % 7) AS DATE) AS week_start
FROM (
  SELECT explode(sequence(DATE '2025-07-01', DATE '2026-06-30', INTERVAL 1 DAY)) AS d
);


In [0]:
%sql
-- R1: claims.analytics.dim_member — member dimension with surrogate key.
-- NO PII columns (name, dob, national_id, phone, email are intentionally excluded).
CREATE OR REPLACE TABLE IDENTIFIER(:catalog ||'.analytics.dim_member')
COMMENT 'R1: member dimension (no PII: name/dob/national_id/phone/email excluded); member_sk surrogate key'
AS
SELECT
  CAST(ROW_NUMBER() OVER (ORDER BY member_id) AS BIGINT) AS member_sk,
  member_id,
  gender,
  city,
  state
FROM IDENTIFIER(:catalog ||'.silver.members');


In [0]:
%sql
-- R1: claims.analytics.dim_hospital — hospital dimension with surrogate key.
CREATE OR REPLACE TABLE IDENTIFIER(:catalog ||'.analytics.dim_hospital')
COMMENT 'R1: hospital dimension; hospital_sk surrogate key'
AS
SELECT
  CAST(ROW_NUMBER() OVER (ORDER BY hospital_id) AS BIGINT) AS hospital_sk,
  hospital_id,
  name      AS hospital_name,
  city,
  city_tier,
  state
FROM IDENTIFIER(:catalog || '.silver.hospitals');


In [0]:
%sql
-- R1: dim_plan — plan attributes per POLICY VERSION (not per plan).
-- Validity windows come from the SCD2 policy history (silver.dim_policy_scd2, the same
-- windows the as-of join in publish_gold uses); plan attributes come from silver.plans via plan_code.
-- valid_to is empty (NULL) while the version is still open; is_current marks those rows.
CREATE OR REPLACE TABLE IDENTIFIER(:catalog ||'.analytics.dim_plan')
COMMENT 'R1: plan dimension, one row per policy version; valid_to empty = still open; is_current flags open versions'
AS
SELECT
  CAST(ROW_NUMBER() OVER (
    ORDER BY pv.policy_id, pv.__START_AT, pv.__END_AT
  ) AS BIGINT)                                         AS plan_sk,
  pv.member_id,
  pv.policy_id,
  pv.plan_code,
  p.plan_family,
  CAST(p.sum_insured AS INT)           AS sum_insured,
  CAST(p.room_rent_cap_per_day AS INT) AS room_rent_cap_per_day,
  CAST(p.annual_premium AS INT)        AS annual_premium,
  CAST(pv.__START_AT AS DATE)          AS valid_from,
  CAST(pv.__END_AT AS DATE)            AS valid_to,
  pv.__END_AT IS NULL                  AS is_current
FROM IDENTIFIER(:catalog || '.silver.dim_policy_scd2') pv
LEFT JOIN IDENTIFIER(:catalog || '.silver.plans') p
  ON p.plan_code = pv.plan_code;


In [0]:
%sql
-- R1: fact_claims — one row per claim with surrogate keys.
-- plan_sk is the PLAN AT ADMISSION (matched on the dim_plan validity window), never the current plan.
-- settled_date_key stays empty until the claim is settled.
CREATE OR REPLACE TABLE IDENTIFIER(:catalog || '.analytics.fact_claims')
COMMENT 'R1: one row per claim; plan_sk = plan at admission; settled_date_key empty until settled'
CLUSTER BY (admission_date_key)
AS
SELECT
  e.claim_id,
  m.member_sk,
  h.hospital_sk,
  p.plan_sk,
  CAST(date_format(e.admission_date, 'yyyyMMdd') AS INT) AS admission_date_key,
  CASE
    WHEN e.submitted_ts IS NULL THEN NULL
    ELSE CAST(date_format(CAST(e.submitted_ts AS DATE), 'yyyyMMdd') AS INT)
  END AS submitted_date_key,
  CASE
    WHEN e.settled_ts IS NULL THEN NULL
    ELSE CAST(date_format(CAST(e.settled_ts AS DATE), 'yyyyMMdd') AS INT)
  END AS settled_date_key,
  e.claim_type,
  e.status,
  e.rejection_code,
  e.is_network_at_admission,
  e.room_rent_over_cap,
  CAST(round(e.billed_amount, 2)   AS DECIMAL(18,2)) AS billed_amount,
  CAST(round(e.approved_amount, 2) AS DECIMAL(18,2)) AS approved_amount,
  e.length_of_stay,
  e.turnaround_days
FROM IDENTIFIER(:catalog || '.gold.claims_enriched') e
LEFT JOIN IDENTIFIER(:catalog || '.analytics.dim_member') m
  ON m.member_id = e.member_id
LEFT JOIN IDENTIFIER(:catalog || '.analytics.dim_hospital') h
  ON h.hospital_id = e.hospital_id
LEFT JOIN IDENTIFIER(:catalog || '.analytics.dim_plan') p
  ON p.policy_id   = e.policy_id
 AND p.plan_code   = e.plan_code_at_admission
 AND p.valid_from <= e.admission_date
 AND (p.valid_to IS NULL OR e.admission_date < p.valid_to);


In [0]:
%sql
-- R1 Keys: named PK/FK constraints (informational, NOT ENFORCED). CTAS cannot declare
-- them inline, so they are added here AFTER the tables are built.
-- NOTE: uses ${catalog} interpolation (python var from cell 1) because :param binding
-- does not parse inside a multi-statement SQL cell.
-- RUN THE FULL CELL (not a selection): FKs are dropped BEFORE the PKs they reference,
-- and every ADD is preceded by DROP CONSTRAINT IF EXISTS, so re-runs are safe.

-- Primary key columns must be NOT NULL before carrying a PK (no-op if already set)
ALTER TABLE ${catalog}.analytics.dim_date     ALTER COLUMN date_key    SET NOT NULL;
ALTER TABLE ${catalog}.analytics.dim_member   ALTER COLUMN member_sk   SET NOT NULL;
ALTER TABLE ${catalog}.analytics.dim_hospital ALTER COLUMN hospital_sk SET NOT NULL;
ALTER TABLE ${catalog}.analytics.dim_plan     ALTER COLUMN plan_sk     SET NOT NULL;
ALTER TABLE ${catalog}.analytics.fact_claims  ALTER COLUMN claim_id    SET NOT NULL;

-- Foreign keys must be dropped BEFORE the PKs they reference (child-constraint error)
ALTER TABLE ${catalog}.analytics.fact_claims DROP CONSTRAINT IF EXISTS fact_claims_member_fk;
ALTER TABLE ${catalog}.analytics.fact_claims DROP CONSTRAINT IF EXISTS fact_claims_hospital_fk;
ALTER TABLE ${catalog}.analytics.fact_claims DROP CONSTRAINT IF EXISTS fact_claims_plan_fk;
ALTER TABLE ${catalog}.analytics.fact_claims DROP CONSTRAINT IF EXISTS fact_claims_admission_date_fk;
ALTER TABLE ${catalog}.analytics.fact_claims DROP CONSTRAINT IF EXISTS fact_claims_submitted_date_fk;
ALTER TABLE ${catalog}.analytics.fact_claims DROP CONSTRAINT IF EXISTS fact_claims_settled_date_fk;

-- Primary keys (named per R1 contract)
ALTER TABLE ${catalog}.analytics.dim_date     DROP CONSTRAINT IF EXISTS dim_date_pk;
ALTER TABLE ${catalog}.analytics.dim_member   DROP CONSTRAINT IF EXISTS dim_member_pk;
ALTER TABLE ${catalog}.analytics.dim_hospital DROP CONSTRAINT IF EXISTS dim_hospital_pk;
ALTER TABLE ${catalog}.analytics.dim_plan     DROP CONSTRAINT IF EXISTS dim_plan_pk;
ALTER TABLE ${catalog}.analytics.fact_claims  DROP CONSTRAINT IF EXISTS fact_claims_pk;
ALTER TABLE ${catalog}.analytics.dim_date     ADD CONSTRAINT dim_date_pk     PRIMARY KEY (date_key);
ALTER TABLE ${catalog}.analytics.dim_member   ADD CONSTRAINT dim_member_pk   PRIMARY KEY (member_sk);
ALTER TABLE ${catalog}.analytics.dim_hospital ADD CONSTRAINT dim_hospital_pk PRIMARY KEY (hospital_sk);
ALTER TABLE ${catalog}.analytics.dim_plan     ADD CONSTRAINT dim_plan_pk     PRIMARY KEY (plan_sk);
ALTER TABLE ${catalog}.analytics.fact_claims  ADD CONSTRAINT fact_claims_pk  PRIMARY KEY (claim_id);

-- Foreign keys (must come after the referenced PKs; named per R1 contract)
ALTER TABLE ${catalog}.analytics.fact_claims ADD CONSTRAINT fact_claims_member_fk
  FOREIGN KEY (member_sk) REFERENCES ${catalog}.analytics.dim_member (member_sk);
ALTER TABLE ${catalog}.analytics.fact_claims ADD CONSTRAINT fact_claims_hospital_fk
  FOREIGN KEY (hospital_sk) REFERENCES ${catalog}.analytics.dim_hospital (hospital_sk);
ALTER TABLE ${catalog}.analytics.fact_claims ADD CONSTRAINT fact_claims_plan_fk
  FOREIGN KEY (plan_sk) REFERENCES ${catalog}.analytics.dim_plan (plan_sk);
ALTER TABLE ${catalog}.analytics.fact_claims ADD CONSTRAINT fact_claims_admission_date_fk
  FOREIGN KEY (admission_date_key) REFERENCES ${catalog}.analytics.dim_date (date_key);
ALTER TABLE ${catalog}.analytics.fact_claims ADD CONSTRAINT fact_claims_submitted_date_fk
  FOREIGN KEY (submitted_date_key) REFERENCES ${catalog}.analytics.dim_date (date_key);
ALTER TABLE ${catalog}.analytics.fact_claims ADD CONSTRAINT fact_claims_settled_date_fk
  FOREIGN KEY (settled_date_key) REFERENCES ${catalog}.analytics.dim_date (date_key);


In [0]:
# R1 verification: row counts, key contract, plan-at-admission correctness
# Uses f-string interpolation (same style as publish_gold) — the catalog widget value
# from cell 1 is substituted before execution.
verify = spark.sql(f"""
  SELECT
    (SELECT COUNT(*) FROM {catalog}.analytics.dim_date)     AS dim_date_rows,      -- expect 365
    (SELECT COUNT(*) FROM {catalog}.analytics.dim_member)   AS dim_member_rows,
    (SELECT COUNT(*) FROM {catalog}.analytics.dim_hospital) AS dim_hospital_rows,
    (SELECT COUNT(*) FROM {catalog}.analytics.dim_plan)     AS dim_plan_rows,      -- one per policy version
    (SELECT COUNT(*) FROM {catalog}.analytics.dim_plan WHERE is_current) AS dim_plan_current,
    (SELECT COUNT(*) FROM {catalog}.analytics.fact_claims)  AS fact_rows,          -- expect = claims_enriched rows
    (SELECT COUNT(*) FROM {catalog}.analytics.fact_claims
       WHERE member_sk IS NULL OR hospital_sk IS NULL OR admission_date_key IS NULL) AS fact_null_keys,   -- expect 0
    (SELECT COUNT(*) FROM {catalog}.analytics.fact_claims f
       JOIN {catalog}.gold.claims_enriched e ON e.claim_id = f.claim_id
       WHERE f.plan_sk IS NULL AND e.plan_code_at_admission IS NOT NULL)         AS fact_unmatched_plan, -- expect 0
    (SELECT COUNT(*) FROM {catalog}.information_schema.key_column_usage
       WHERE table_schema = 'analytics' AND constraint_name LIKE '%_pk')    AS pk_constraints,     -- expect 5
    (SELECT COUNT(*) FROM {catalog}.information_schema.table_constraints
       WHERE table_schema = 'analytics' AND constraint_name LIKE 'fact_claims_%_fk') AS fk_constraints -- expect 6
""")
display(verify)

spot = spark.sql(f"""
  -- plan_sk must resolve to the version whose window contains the admission date,
  -- NOT the current plan, for policies that changed plan mid-term.
  SELECT f.claim_id, e.admission_date, e.plan_code_at_admission,
         dp.plan_code AS dim_plan_code, dp.valid_from, dp.valid_to
  FROM {catalog}.analytics.fact_claims f
  JOIN {catalog}.analytics.dim_plan dp ON dp.plan_sk = f.plan_sk
  JOIN {catalog}.gold.claims_enriched e ON e.claim_id = f.claim_id
  LIMIT 10
""")
display(spot)
